# 3 ~ VAE y el score recon_prob

A diferencia del AE, el decoder del VAE predice dos cosas por feature: una media mu_x (la reconstrucción en sí) y una varianza logvar_x (qué tan seguro está el modelo de esa reconstrucción). Eso habilita distintas formas de convertir el error de reconstrucción en un score de anomalía:

- recon_error (MSE): el error cuadrático plano entre entrada y reconstrucción, igual que en el AE. Trata todas las features por igual, sin usar la varianza que predice el decoder — dos features pueden tener el mismo error absoluto y contar lo mismo, aunque una sea intrínsecamente más ruidosa que la otra.

- recon_prob (An & Cho 2015) — la clave del proyecto: en vez de MSE plano, calcula -E[log p(x|z)], la log-verosimilitud negativa gaussiana de cada feature, ponderada por su propia varianza predicha: (x - mu_x)² / exp(logvar_x). Esto significa que el error en una feature donde el decoder predijo alta varianza (poca confianza) pesa menos, mientras que el error en una feature donde predijo baja varianza (mucha confianza) pesa más — un desvío "inesperado" en algo que el modelo creía tener bien entendido es más sospechoso que un desvío en algo naturalmente ruidoso. Se estima con Monte Carlo, promediando sobre varias muestras del espacio latente.

- neg_elbo: el mismo término de recon_prob, pero sumándole la divergencia KL entre q(z|x) y el prior p(z), ponderada por beta (β-VAE). Es la métrica que se usa para entrenar (o una aproximación de un solo paso), pero como score de anomalía mezcla error de reconstrucción con regularización del espacio latente — no es puramente "qué tan mal se reconstruyó".

In [50]:
import lab                      # utilidades: métricas y gráficos (experiments/lab.py)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, data
from src.models import (IsolationForestDetector, OneClassSVMDetector,
                        ZScoreDetector, MahalanobisDetector,
                        AEDetector, DenoisingAEDetector, VAEDetector,
                        EnsembleDetector, DeepODDetector)
plt.rcParams["figure.dpi"] = 110
SEEDS = lab.SEEDS            # 10 semillas del estudio. Bajalas (p. ej. [42,43,44])
print("semillas:", SEEDS)   # para un Run all más rápido; los números se mueven ±std

semillas: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


In [51]:
panel_z = data.prepare()                         # panel + etiqueta z_rinde
ds   = data.build_crop_dataset(panel_z, "soja")  # split + normalización (soja)
dsm  = data.build_crop_dataset(panel_z, "maiz")  # idem maíz
print("soja  train:", ds.X_train.shape, "| test anómalas:", int(ds.y_test.sum()))
print("maíz  train:", dsm.X_train.shape, "| test anómalas:", int(dsm.y_test.sum()))

[data] dedup panel: 27865 -> 20672 filas (7193 duplicados espurios por lat/lon eliminados)
soja  train: (6143, 72) | test anómalas: 251
maíz  train: (8082, 72) | test anómalas: 344


## 3.1 ~ Búsqueda de arquitectura 
Barremos configuraciones de arquitectura/regularización del VAE (`hidden_dims`,
`latent_dim`, `beta`), con `score_mode=recon_prob` fijo. 


In [52]:
datasets = {"soja": ds, "maiz": dsm}
candidatas = [((64,32),16,1.0), ((64,32),24,1.0), ((64,32),24,0.5),
              ((128,64),16,1.0), ((128,64),24,1.0), ((128,64),24,0.5)]
filas_hp = []
for cultivo, dset in datasets.items():
    for hid, lat, beta in candidatas:
        prs = []
        for s in SEEDS[:5]:                                # 5 semillas por configuración
            v = VAEDetector(hidden_dims=hid, latent_dim=lat, beta=beta, score_mode="recon_prob",
                            n_mc_samples=50, max_epochs=300, patience=30, random_state=s).fit(dset.X_train)
            prs.append(lab.metrics(v.score_samples(dset.X_test), dset.y_test)["pr_auc"])
        prs = np.array(prs)
        filas_hp.append({"cultivo": cultivo, "hidden_dims": str(hid), "latent_dim": lat, "beta": beta,
                         "pr_auc_1seed": round(prs[0],3), "pr_auc_mean": round(prs.mean(),3),
                         "pr_auc_std": round(prs.std(),3), "media_menos_std": round(prs.mean()-prs.std(),3)})
hp = pd.DataFrame(filas_hp).sort_values(["cultivo", "media_menos_std"], ascending=[True, False]).reset_index(drop=True)
hp

,cultivo,hidden_dims,latent_dim,beta,pr_auc_1seed,pr_auc_mean,pr_auc_std,media_menos_std
0,maiz,"(128, 64)",16,1.0,0.512,0.535,0.030,0.505
1,maiz,"(128, 64)",24,1.0,0.553,0.511,0.031,0.480
2,maiz,"(64, 32)",24,1.0,0.474,0.529,0.059,0.470
3,maiz,"(128, 64)",24,0.5,0.522,0.505,0.035,0.470
4,maiz,"(64, 32)",24,0.5,0.496,0.505,0.037,0.467
5,maiz,"(64, 32)",16,1.0,0.410,0.509,0.056,0.453
6,soja,"(128, 64)",16,1.0,0.560,0.622,0.033,0.590
7,soja,"(64, 32)",16,1.0,0.567,0.556,0.009,0.547
8,soja,"(128, 64)",24,1.0,0.592,0.599,0.053,0.546
9,soja,"(64, 32)",24,1.0,0.641,0.573,0.056,0.517


Se elige el mejor modelo, aquel que tiene mayor media_menos_std. Esto es porque queremos el modelo que tenga mayor PR-AUC medio, pero con menor varianza (no "mejor" varianza — más std siempre es peor). Restar el std penaliza a los modelos que ganan por suerte en pocas semillas: un modelo con media alta pero std alto no es confiable, aunque su promedio sea el más alto.

In [53]:
BEST = {}
for cultivo in datasets:
    best = hp[hp["cultivo"] == cultivo].iloc[0]
    BEST[cultivo] = dict(hidden_dims=eval(best["hidden_dims"]), latent_dim=int(best["latent_dim"]),
                         beta=float(best["beta"]))
    print(f"config final {cultivo} (mejor media−std): {BEST[cultivo]}")
    print(f"  PR-AUC 1 semilla = {best['pr_auc_1seed']}  vs  media±std = {best['pr_auc_mean']}±{best['pr_auc_std']}")

config final soja (mejor media−std): {'hidden_dims': (128, 64), 'latent_dim': 16, 'beta': 1.0}
  PR-AUC 1 semilla = 0.56  vs  media±std = 0.622±0.033
config final maiz (mejor media−std): {'hidden_dims': (128, 64), 'latent_dim': 16, 'beta': 1.0}
  PR-AUC 1 semilla = 0.512  vs  media±std = 0.535±0.03


## 3.2 ~ Los tres scores del VAE, con la misma red (config ganadora de soja)
`recon_error` (MSE), `neg_elbo` (recon + KL) y `recon_prob`. Entrenamos una vez por
semilla y evaluamos los tres scores del mismo modelo — así la comparación aísla el
*score*. Diagnóstico ilustrativo sobre soja; el mismo patrón vale para maíz.

In [54]:
filas = {"recon_error": [], "neg_elbo": [], "recon_prob": []}
for seed in SEEDS:
    vae = VAEDetector(**BEST["soja"], n_mc_samples=50, score_mode="recon_prob",
                      max_epochs=300, patience=30, random_state=seed).fit(ds.X_train)
    for modo in filas:
        vae.score_mode = modo                                  # mismo modelo, distinto score
        filas[modo].append(lab.metrics(vae.score_samples(ds.X_test), ds.y_test))
pd.DataFrame({modo: lab.mean_std(f) for modo, f in filas.items()}).T[["pr_auc","roc_auc","recall_at_contam"]]

,pr_auc,roc_auc,recall_at_contam
recon_error,0.272±0.042,0.511±0.066,0.090±0.042
neg_elbo,0.598±0.041,0.749±0.030,0.295±0.012
recon_prob,0.598±0.041,0.749±0.031,0.295±0.012


El usar MSE para la reconstrucción resultó, como ya se vio en el AE y el DAE, desastroso. En cambio, al utilizar reconstrucciones probabilísticas —recon_prob y neg_elbo— la diferencia es enorme. Ambos métodos dan resultados parecidos, pero se continúa con recon_prob por ser el estimador que la literatura de detección de anomalías definió específicamente para este propósito (An & Cho 2015); neg_elbo, en cambio, es el objetivo de entrenamiento del VAE (la ELBO) reutilizado como score, no un estimador pensado para anomalías.

Existe un sesgo de elección: la arquitectura se eligió usando recon_prob como métrica, lo que puede favorecerlo frente a recon_error en esta comparación. Sin embargo, la brecha entre ambos es demasiado grande para explicarse solo por eso — es una limitación estructural del MSE (trata todas las features por igual, sin ponderar por la confianza del decoder), no un detalle de la arquitectura.

In [55]:
filas_vae_soja = filas["recon_prob"]
print("VAE recon_prob (soja):"); display(lab.mean_std(filas_vae_soja))

filas_vae_maiz = []
for seed in SEEDS:
    v = VAEDetector(**BEST["maiz"], n_mc_samples=50, score_mode="recon_prob",
                    max_epochs=300, patience=30, random_state=seed).fit(dsm.X_train)
    filas_vae_maiz.append(lab.metrics(v.score_samples(dsm.X_test), dsm.y_test))
print("\nVAE recon_prob (maíz):"); display(lab.mean_std(filas_vae_maiz))

VAE recon_prob (soja):


pr_auc              0.598±0.041
roc_auc             0.749±0.031
prec_at_k           0.568±0.048
recall_at_contam    0.295±0.012
dtype: str


VAE recon_prob (maíz):


pr_auc              0.518±0.041
roc_auc             0.668±0.036
prec_at_k           0.472±0.047
recall_at_contam    0.242±0.019
dtype: str

## 3.3 ~La regularización pesada destruye la señal


In [ ]:
regularizacion_vae = [
    {"beta": 1.0,  "dropout": 0.0,  "use_batch_norm": False, "weight_decay": 0.0},
    {"beta": 0.5,  "dropout": 0.0,  "use_batch_norm": False, "weight_decay": 0.0},
    {"beta": 0.5,  "dropout": 0.15, "use_batch_norm": True,  "weight_decay": 0.0},
    {"beta": 0.5,  "dropout": 0.15, "use_batch_norm": True,  "weight_decay": 1e-4},
    {"beta": 0.1,  "dropout": 0.0,  "use_batch_norm": False, "weight_decay": 0.0},
    {"beta": 0.1,  "dropout": 0.15, "use_batch_norm": True,  "weight_decay": 0.0},
]

resultados_reg_vae = {}
for reg in regularizacion_vae:
    nombre = f"beta{reg['beta']}_do{reg['dropout']}_bn{reg['use_batch_norm']}_wd{reg['weight_decay']}"

    def make_detector(seed, reg=reg):
        return VAEDetector(
            hidden_dims=BEST["soja"]["hidden_dims"], latent_dim=BEST["soja"]["latent_dim"],
            score_mode="recon_prob", n_mc_samples=50,
            beta=reg["beta"], dropout=reg["dropout"],
            use_batch_norm=reg["use_batch_norm"], weight_decay=reg["weight_decay"],
            max_epochs=300, patience=30, random_state=seed,
        )
    mean, std = lab.evaluate(make_detector, ds, lab.SEEDS, metric="pr_auc")
    resultados_reg_vae[nombre] = (mean, std)

tabla_reg = pd.DataFrame(
    [{"config": n, "pr_auc_mean": m, "pr_auc_std": s} for n, (m, s) in resultados_reg_vae.items()]
).sort_values("pr_auc_mean", ascending=False)
tabla_reg

la configuración ganadora es la "limpia" — beta=1.0, sin dropout, sin batch norm y sin weight decay (PR-AUC 0.586±0.035) — por encima de todas las variantes regularizadas. Bajar beta a 0.1 ya empeora el resultado (0.540±0.038), y agregar batch norm + dropout lo empeora todavía más, cayendo hasta 0.47-0.49 en las tres configs con bn=True. Esto confirma que ninguna regularización adicional ayuda: el VAE ya tiene su propio mecanismo de regularización natural (la varianza per-feature que pondera recon_prob), y sumarle capas de regularización extra solo aplana la señal en vez de mejorarla.

## 3.4 · VAE vs TODOS los baselines
Contra el conjunto completo del nb 1 — estadísticos (z-score, Mahalanobis) y de modelos
(IForest, OCSVM). Todos multi-seed donde corresponde (IForest estocástico → ±std; los
deterministas → std 0):

In [ ]:
resultados = {
    "ZScore (mean)": [lab.metrics(ZScoreDetector(agg="mean").fit(ds.X_train)
                                  .score_samples(ds.X_test), ds.y_test)],
    "Mahalanobis":   [lab.metrics(MahalanobisDetector(shrinkage=None).fit(ds.X_train)
                                  .score_samples(ds.X_test), ds.y_test)],
    "IForest": [lab.metrics(IsolationForestDetector(n_estimators=200, max_features=0.3, random_state=s)
                            .fit(ds.X_train).score_samples(ds.X_test), ds.y_test) for s in SEEDS],
    "OCSVM-RBF (determinista)": [lab.metrics(OneClassSVMDetector(kernel="rbf", nu=0.1)
                              .fit(ds.X_train).score_samples(ds.X_test), ds.y_test)],
    "VAE recon_prob": filas_vae_soja,
}
lab.plot_leaderboard(lab.leaderboard(resultados)); plt.show()
lab.tabla(resultados)

## 3.5 · IForest y OCSVM sobre el latente del VAE (multi-seed, soja y maíz)
Como con el AE, probamos correr detectores shallow sobre el latente del VAE. **En igualdad
de condiciones que el resto**: 5 semillas, media ± std, por cultivo (cada uno con su config
ganadora `BEST[cultivo]`). Para cada semilla entrenamos el VAE, sacamos su latente y
corremos los detectores ahí; guardamos uno como representante por cultivo para el t-SNE
de abajo:

In [ ]:
repr_ = {}   # cultivo -> (vae, Zte, scores) del representante (seed = SEEDS[0])
resumen_lat = {}
for cultivo, dset in datasets.items():
    filas_lat = {"VAE recon_prob (directo)": [], "IForest sobre latente": [], "OCSVM sobre latente": []}
    for s in SEEDS:
        v = VAEDetector(**BEST[cultivo], n_mc_samples=50, score_mode="recon_prob",
                        max_epochs=300, patience=30, random_state=s).fit(dset.X_train)
        Ztr, Zte = v.encode(dset.X_train), v.encode(dset.X_test)
        filas_lat["VAE recon_prob (directo)"].append(lab.metrics(v.score_samples(dset.X_test), dset.y_test)["pr_auc"])
        filas_lat["IForest sobre latente"].append(lab.metrics(IsolationForestDetector(n_estimators=200,
            max_features=0.3, random_state=s).fit(Ztr).score_samples(Zte), dset.y_test)["pr_auc"])
        filas_lat["OCSVM sobre latente"].append(lab.metrics(OneClassSVMDetector(kernel="rbf", nu=0.1)
            .fit(Ztr).score_samples(Zte), dset.y_test)["pr_auc"])
        if s == SEEDS[0]:
            repr_[cultivo] = (v, Zte, v.score_samples(dset.X_test))   # para el t-SNE
    resumen_lat[cultivo] = pd.Series({k: f"{np.mean(v):.3f}±{np.std(v):.3f}" for k, v in filas_lat.items()})

pd.DataFrame(resumen_lat)

**El `recon_prob` directo gana** (comparando las medias ± std): el latente pierde la
señal que el score probabilístico captura (el error ponderado por varianza vive en el
espacio de reconstrucción, no en el latente). Un detector shallow sobre el latente no la
recupera — en ninguno de los dos cultivos.

## 3.6 ~ t-SNE del espacio latente (soja y maíz)
Proyectamos el latente del VAE representativo de cada cultivo a 2D con t-SNE, coloreado por
etiqueta real y por score. Si las anomalías no forman cluster, ningún modelo que mire este
espacio las separa caso a caso (anticipo del techo, nb 5):

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11, 9))
for i, cultivo in enumerate(datasets):
    _, Zte_repr, sc_repr = repr_[cultivo]
    emb = lab.embed_2d(Zte_repr, method="tsne")
    lab.plot_embedding(emb, datasets[cultivo].y_test, kind="label", ax=axs[i, 0],
                       titulo=f"t-SNE latente · etiqueta real ({cultivo})")
    lab.plot_embedding(emb, sc_repr, kind="score", ax=axs[i, 1],
                       titulo=f"t-SNE latente · score del VAE ({cultivo})")
plt.tight_layout(); plt.show()

**Mejor modelo hasta acá: VAE `recon_prob`** — supera al IForest y al OCSVM, en soja y en
maíz. Pero tiene **alta varianza entre semillas** (ver el ±std de arriba): en una corrida
mala roza el baseline. Lo resolvemos con el ensemble (nb 4), reusando `BEST["soja"]` y
`BEST["maiz"]` respectivamente.